# FreightBid Phase 1 Experiments

Ablations over scoring weights + visualizations of plan quality.

In [ ]:
import json
from pathlib import Path
from datetime import datetime

from adapters.inbound.api.container import build_container
from adapters.inbound.api.mappers import load_from_dto, truck_from_dto
from adapters.inbound.api.schemas import LoadDTO, TruckStateDTO

ROOT = Path('..').resolve()
c = build_container(ROOT / 'config')

loads_payload = json.loads((ROOT / 'sample_data' / 'loads.json').read_text())
truck_payload = json.loads((ROOT / 'sample_data' / 'truck.json').read_text())
loads = [load_from_dto(LoadDTO(**l)) for l in loads_payload['loads']]
truck = truck_from_dto(TruckStateDTO(**truck_payload))

ranked = c.recommender.recommend_loads(loads, truck, top_n=10)
for ev, sc in ranked:
    print(sc.load_id, round(sc.score,2), round(sc.expected_profit,2), sc.rationale)

plan = c.planner.build_plan(loads, truck)
print('PLAN', plan.load_ids, round(plan.expected_profit,2))


## Ablation: profit-only vs full heuristic

In [ ]:
from dataclasses import replace
from domain.scoring.heuristic_scoring import HeuristicScoringStrategy
from application.plan_builder import PlanBuilderService

variants = {
    'profit_only': replace(c.config.scoring_weights, rate_per_mile_weight=0, deadhead_miles_penalty=0, driver_hours_penalty=0),
    'baseline':    c.config.scoring_weights,
}
for name, w in variants.items():
    strategy = HeuristicScoringStrategy(w, c.config.cost_model)
    planner = PlanBuilderService(strategy, c.config.planning_constraints, c.evaluator)
    p = planner.build_plan(loads, truck)
    print(name, p.load_ids, round(p.expected_profit,2), round(p.expected_deadhead_miles,1))
